In [ ]:
import os
import torch
import torchaudio
import pandas as pd
import pickle

from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification
)

In [ ]:
# Device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Emotion mapping

emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fear",
    "07": "disgust",
    "08": "surprise"
}

In [ ]:
# Load dataset

data = []

base_path = "../datasets/ravdess"

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".wav"):

            emotion_code = file.split("-")[2]
            emotion = emotion_map[emotion_code]

            path = os.path.join(root, file)

            data.append([path, emotion])

df = pd.DataFrame(data, columns=["path", "emotion"])


In [ ]:
# Encode labels

le = LabelEncoder()
df["label"] = le.fit_transform(df["emotion"])

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

In [ ]:
# Train / Validation split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)    

In [ ]:
# Processor

processor = Wav2Vec2Processor.from_pretrained(
    "facebook/wav2vec2-base"
)

In [ ]:
# Dataset class

class RAVDESSDataset(torch.utils.data.Dataset):

    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        path = self.data.loc[idx, "path"]
        label = self.data.loc[idx, "label"]

        speech, sr = torchaudio.load(path)

        speech = speech.squeeze()

        # limit audio length (important for memory)
        speech = speech[:16000*4]

        speech = speech.numpy()

        inputs = processor(
            speech,
            sampling_rate=sr,
            return_tensors="pt",
            padding=True
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }

In [ ]:
# DataLoaders

train_dataset = RAVDESSDataset(train_df)
val_dataset = RAVDESSDataset(val_df)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=4
)

In [ ]:
# Load model

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=8
)

# Freeze feature encoder (saves GPU memory)
model.freeze_feature_encoder()

model = model.to(device)


In [ ]:
# Optimizer

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5
)

loss_fn = torch.nn.CrossEntropyLoss()


In [ ]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0

    for batch in tqdm(train_loader):

        optimizer.zero_grad()

        input_values = batch["input_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_values)

        logits = outputs.logits

        loss = loss_fn(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()

    acc = correct / len(train_dataset)

    print("Epoch:", epoch+1)
    print("Train Accuracy:", acc)